# Tech Challenge Fase 2
## 03.2 — Gold Municípios

Integra:

- indicadores Silver de municípios;
- metas municipais;
- indicadores agregados da Gold Alunos.

O produto final oferece uma visão municipal enriquecida.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Leitura das bases

## Garantia da granularidade antes dos joins

Cada fonte é reduzida previamente para uma linha por:

```text
ANO + CO_UF + CO_MUNICIPIO
```

Isso evita relações muitos-para-muitos e garante que a Gold Municípios permaneça única por município e ano.

In [0]:
metadata = pd.read_parquet(
    CONFIG_PATH / "gold_metadata"
)

bases = []

def primeira_coluna_disponivel(df, candidatas):
    for coluna in candidatas:
        if coluna in df.columns:
            return coluna
    return None


def consolidar_por_municipio(df, ano, origem):
    df = df.copy()
    df["ANO"] = ano

    if "CO_MUNICIPIO" not in df.columns:
        raise KeyError(
            f"{origem} {ano}: CO_MUNICIPIO ausente."
        )

    if "CO_UF" not in df.columns:
        raise KeyError(
            f"{origem} {ano}: CO_UF ausente."
        )

    df["CO_MUNICIPIO"] = normalizar_codigo(
        df["CO_MUNICIPIO"]
    )
    df["CO_UF"] = normalizar_codigo(
        df["CO_UF"]
    )

    df = df[
        df["CO_MUNICIPIO"].notna()
        & ~df["CO_MUNICIPIO"].astype(str).str.lower().isin(
            ["", "nan", "none", "<na>", "null"]
        )
    ].copy()

    chaves = ["ANO", "CO_UF", "CO_MUNICIPIO"]

    colunas_numericas = [
        coluna for coluna in df.columns
        if coluna not in chaves
        and pd.api.types.is_numeric_dtype(df[coluna])
    ]

    colunas_descritivas = [
        coluna for coluna in df.columns
        if coluna not in chaves
        and coluna not in colunas_numericas
    ]

    agregacoes = {
        coluna: "mean"
        for coluna in colunas_numericas
    }

    agregacoes.update({
        coluna: "first"
        for coluna in colunas_descritivas
    })

    if agregacoes:
        df = (
            df.groupby(
                chaves,
                dropna=False
            )
            .agg(agregacoes)
            .reset_index()
        )
    else:
        df = (
            df[chaves]
            .drop_duplicates()
            .reset_index(drop=True)
        )

    if df.duplicated(
        subset=["ANO", "CO_MUNICIPIO"]
    ).any():
        raise ValueError(
            f"{origem} {ano}: duplicidade após consolidação."
        )

    return df


for ano in [2023, 2024, 2025]:
    meta_mun = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    meta_metas = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "metas_municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    df_mun = ler_csv(
        Path(meta_mun["silver_path"])
        / meta_mun["silver_file_name"]
    )

    df_metas = ler_csv(
        Path(meta_metas["silver_path"])
        / meta_metas["silver_file_name"]
    )

    df_alunos = ler_csv(
        GOLD_PATH
        / "alunos"
        / f"ano={ano}"
        / f"GOLD_ALUNOS_{ano}.csv"
    )

    df_mun = consolidar_por_municipio(
        df_mun,
        ano,
        "municipios"
    )

    df_metas = consolidar_por_municipio(
        df_metas,
        ano,
        "metas_municipios"
    )

    df_alunos = consolidar_por_municipio(
        df_alunos,
        ano,
        "gold_alunos"
    )

    # Mantém os atributos descritivos da base de municípios.
    df_metas_join = df_metas.drop(
        columns=[
            coluna for coluna in [
                "SG_UF",
                "NO_MUNICIPIO"
            ]
            if coluna in df_metas.columns
        ],
        errors="ignore"
    )

    df_alunos_join = df_alunos.drop(
        columns=[
            coluna for coluna in [
                "SG_UF",
                "NO_MUNICIPIO"
            ]
            if coluna in df_alunos.columns
        ],
        errors="ignore"
    )

    df_integrado = (
        df_mun
        .merge(
            df_metas_join,
            on=["ANO", "CO_UF", "CO_MUNICIPIO"],
            how="left",
            validate="one_to_one",
            suffixes=("", "_META")
        )
        .merge(
            df_alunos_join,
            on=["ANO", "CO_UF", "CO_MUNICIPIO"],
            how="left",
            validate="one_to_one",
            suffixes=("", "_ALUNOS")
        )
    )

    if df_integrado.duplicated(
        subset=["ANO", "CO_MUNICIPIO"]
    ).any():
        raise ValueError(
            f"Gold Municípios {ano}: duplicidade após joins."
        )

    bases.append(df_integrado)

df_gold_municipios = pd.concat(
    bases,
    ignore_index=True
)

if df_gold_municipios.duplicated(
    subset=["ANO", "CO_MUNICIPIO"]
).any():
    raise ValueError(
        "Gold Municípios consolidada possui duplicidades."
    )

print(
    "Granularidade Gold Municípios validada: "
    "uma linha por ANO + CO_MUNICIPIO."
)

## 5. Indicadores e risco

In [0]:
for coluna in ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP", "META_FINAL_2030"]:
    if coluna in df_gold_municipios.columns:
        df_gold_municipios[coluna] = converter_numero(df_gold_municipios[coluna])

df_gold_municipios["gap_meta_2030"] = (
    df_gold_municipios["META_FINAL_2030"]
    - df_gold_municipios["PC_ALUNO_ALFABETIZADO"]
)

df_gold_municipios["risco_educacional"] = pd.cut(
    df_gold_municipios["PC_ALUNO_ALFABETIZADO"],
    bins=[-np.inf, 50, 70, 85, np.inf],
    labels=["Crítico", "Alto", "Médio", "Baixo"]
)

df_gold_municipios["_gold_processed_at"] = datetime.now().isoformat()

if df_gold_municipios.empty:
    print("Gold Municípios sem registros.")
else:
    display(df_gold_municipios.head())

## 6. Persistência

In [0]:
for ano in [2023, 2024, 2025]:
    registro = metadata[
        (metadata["produto"] == "gold_municipios")
        & (metadata["dataset"] == "municipios")
        & (metadata["ano"] == ano)
    ].iloc[0]

    salvar_csv(
        df_gold_municipios[df_gold_municipios["ANO"] == ano],
        Path(registro["gold_output_path"]),
        registro["gold_file_name"]
    )

print("Gold Municípios salva com sucesso.")